In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.stats
from matplotlib.patches import Patch
from tqdm.contrib.concurrent import thread_map

from lm import LM  # api LMs via OpenRouter, caches answers locally

# Constants

In [ ]:
DATASET_PATH = Path("~/data").expanduser()  # will make directory if necessary

LMS = {
    "gemini": LM("gemini-3-pro-preview"),
    "gpt": LM("gpt-5"),
    "grok": LM("grok-4"),
    "opus": LM("opus-4.5"),
}


LM_NAMES = {
    "gemini": "LLM1",
    "gpt": "LLM2",
    "grok": "LLM3",
    "opus": "LLM4",
}

DEFAULT_SYSTEM_MESSAGE = "You are a helpful assistant."

ABS_REWARDS = [0, 0.5, 0.75, 0.9]  # abstention rewards, correspond to penalties of 0, 1, 3, 9
MAX_PARALLEL = 200  # threads
SEED = 0

ABSTAIN_ADDENDUM0 = (
    "[Never abstain. Make your best guess if unsure. Abstention is scored as incorrect.]"
)
ABSTAIN_ADDENDUM_TEMPLATE = "[Abstain if probability of being correct is < {percent}. Abstention is scored at {percent} correct.]"

In [ ]:
# SimpleQA constants

NUM_SAMPLES = (
    None  # can restrict the number of samples to 3 for making sure it works, None is all samples
)

GRADER_LM = LM("gpt-4.1")  # the standard grader used for SimpleQA

# https://github.com/openai/simple-evals/blob/ee3b0318d8d1d9d72755a4120879be65f7c07e9e/simpleqa_eval.py#L13
SIMPLEQA_GRADER_TEMPLATE = """
Your job is to look at a question, a gold target, and a predicted answer, and then assign a grade of either ["CORRECT", "INCORRECT", "NOT_ATTEMPTED"].
First, I will give examples of each grade, and then you will grade a new example.


The following are examples of CORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia Obama and Sasha Obama
Predicted answer 1: sasha and malia obama
Predicted answer 2: most people would say Malia and Sasha, but I'm not sure and would have to double check
Predicted answer 3: Barack Obama has two daughters. Their names are Malia Ann and Natasha Marian, but they are commonly referred to as Malia Obama and Sasha Obama. Malia was born on July 4, 1998, and Sasha was born on June 10, 2001.
```
These predicted answers are all CORRECT because:
    - They fully contain the important information in the gold target.
    - They do not contain any information that contradicts the gold target.
    - Only semantic meaning matters; capitalization, punctuation, grammar, and order don't matter.
    - Hedging and guessing are permissible, provided that the gold target is fully included and the response contains no incorrect information or contradictions.


The following are examples of INCORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: Malia.
Predicted answer 2: Malia, Sasha, and Susan.
Predicted answer 3: Barack Obama does not have any children.
Predicted answer 4: I think it's either Malia and Sasha. Or it could be Malia and Jackie. Or it could be Joey and Malia.
Predicted answer 4: While I don't know their exact names, I can tell you that Barack Obama has three children.
Predicted answer 5: It's possible you may mean Betsy and Olivia. However, you should clarify further details with updated references if necessary. Is that the correct answer?
Predicted answer 6: It may be the case that Obama's child is named James. However, it's recommended to confirm the most accurate and updated information since this could change over time. This model may not always reflect the most current information.
```
These predicted answers are all INCORRECT because:
    - A factual statement in the answer contradicts the gold target. Incorrect statements that have some hedging (e.g., "it is possible that", "although i'm not sure, i think") are also considered incorrect.


The following are examples of NOT_ATTEMPTED predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: I don't know.
Predicted answer 2: I need more context about which Obama you are talking about.
Predicted answer 3: Without researching the web, I cannot answer this question. However, I can tell you that Barack Obama has two children.
Predicted answer 4: Barack Obama has two children. I know that one of them is Malia, but I'm not sure about the other one.
```
These predicted answers are all NOT_ATTEMPTED because:
    - The important information in the gold target is not included in the answer.
    - No statements in the answer contradict the gold target.


Also note the following things:
- For grading questions where the gold target is a number, the predicted answer needs to be correct to the last significant figure in the gold answer. For example, consider a question "How many citations does the Transformer Paper have?" with gold target "120k". 
    - Predicted answers "120k", "124k", and 115k" are all CORRECT. 
    - Predicted answers "100k" and "113k" are INCORRECT. 
    - Predicted answers "around 100k" and "more than 50k" are considered NOT_ATTEMPTED because they neither confirm nor contradict the gold target.
- The gold target may contain more information than the question. In such cases, the predicted answer only needs to contain the information that is in the question.
    - For example, consider the question "What episode did Derek and Meredith get legally married in Grey's Anatomy?" with gold target "Season 7, Episode 20: White Wedding". Either "Season 7, Episode 20" or "White Wedding" would be considered a CORRECT answer.
- Do not punish predicted answers if they omit information that would be clearly inferred from the question.
    - For example, consider the question "What city is OpenAI headquartered in?" and the gold target "San Francisco, California". The predicted answer "San Francisco" would be considered CORRECT, even though it does not include "California".
    - Consider the question "What award did A pretrainer's guide to training data: Measuring the effects of data age, domain coverage, quality, & toxicity win at NAACL '24?", the gold target is "Outstanding Paper Award". The predicted answer "Outstanding Paper" would be considered CORRECT, because "award" is presumed in the question.
    - For the question "What is the height of Jason Wei in meters?", the gold target is "1.73 m". The predicted answer "1.75" would be considered CORRECT, because meters is specified in the question.
    - For the question "What is the name of Barack Obama's wife?", the gold target is "Michelle Obama". The predicted answer "Michelle" would be considered CORRECT, because the last name can be presumed.
- Do not punish for typos in people's name if it's clearly the same name. 
    - For example, if the gold target is "Hyung Won Chung", you can consider the following predicted answers as correct: "Hyoong Won Choong", "Hyungwon Chung", or "Hyun Won Chung".


Here is a new example. Simply reply with either CORRECT, INCORRECT, NOT ATTEMPTED. Don't apologize or correct yourself if there was a mistake; we are just trying to grade the answer.
```
Question: {question}
Gold target: {target}
Predicted answer: {predicted_answer}
```

Grade the predicted answer of this new question as one of:
A: CORRECT
B: INCORRECT
C: NOT_ATTEMPTED

Just return the letters "A", "B", or "C", with no text around it.
""".strip()


def grade_answer(question, predicted_answer, target, max_tries=3):
    """
    Uses strict grading, retries 3 times if the response is not one of A, B, or C.
    """
    prompt = SIMPLEQA_GRADER_TEMPLATE.format(
        question=question, target=target, predicted_answer=predicted_answer
    )
    for i in range(max_tries):
        res = GRADER_LM.generate(
            prompt=prompt,
            max_tokens=16,
            temperature=0,
            output_mode="with_usage",
            seed=i or None,
        )[0]  # ignore usage
        if res in ["A", "B", "C"]:
            return {"A": "CORRECT", "B": "INCORRECT", "C": "NOT_ATTEMPTED"}[res]
        msg = f"Got weird grade {repr(res)} not in ['A', 'B', 'C'] for:\n{predicted_answer=}\n{question=}\n{target=}"
        # print(msg, flush=True)
    assert False, msg

# Download SimpleQA Data

In [ ]:
# make DATASET_PATH if necessary
if not DATASET_PATH.exists():
    print(f"Making {DATASET_PATH} because it doesn't exist")
    DATASET_PATH.mkdir(parents=True)
simpleqa_path = DATASET_PATH / "simple_qa_test_set.csv"
if not simpleqa_path.exists():
    print(f"Downloading {simpleqa_path}")

    url = "https://openaipublic.blob.core.windows.net/simple-evals/simple_qa_test_set.csv"
    r = requests.get(url)
    open(simpleqa_path, "wb").write(r.content)
df = pd.read_csv(simpleqa_path)
all_simpleqa = []
for _, row in df.iterrows():
    all_simpleqa.append({"problem": row["problem"].strip(), "answer": row["answer"].strip()})

permuted_data = all_simpleqa.copy()
random.Random(17).shuffle(permuted_data)
simpleqa = [{"problem": d["problem"], "answer": d["answer"]} for d in permuted_data[:NUM_SAMPLES]]
print(f"Loaded {len(simpleqa):,} SimpleQA questions")

# Utilities

In [ ]:
def my_map(f, xs, max_workers=MAX_PARALLEL):
    return thread_map(lambda x: f(*x), xs, max_workers=max_workers)


def avg(iterable):
    """Useful helper for averaging."""
    li = list(iterable)
    return sum(li) / len(li)

# Generating responses

* `STATED` is the list of possible stated abstention thresholds
* `k` is the number of responses generated per question
    * `k=1` is the baseline condition
    * `k=2` is the mitigated condition


In [ ]:
STATED = [None] + ABS_REWARDS  # stated = None means unspecified abstention reward

def confidence_string(confidence: float) -> str:
    percent = int(round(100 * confidence, 0))
    return f"{percent}%"


def add_rubric(prompt: str, stated: float | None):
    if stated is None:
        return prompt
    assert stated >= 0
    if stated == 0:
        addendum = ABSTAIN_ADDENDUM0
    else:
        addendum = ABSTAIN_ADDENDUM_TEMPLATE.format(percent=confidence_string(stated))
    # addendum = addendum.replace("0 for no answer", " for no answer") # remove trailing 0 after decimal
    return f"{prompt.rstrip()}\n\n{addendum}"


for stated in STATED:
    print(f"====> {stated=}:\n{add_rubric("What's 1+1?", stated)}\n")


In [ ]:
def answer_question(question, lm: LM, seed=None):
    """Adds system message and generates a response."""
    # system_message = OPENAI_SYSTEM_MESSAGE
    system_message = DEFAULT_SYSTEM_MESSAGE
    return lm.generate(
        prompt=question, system_message=system_message, seed=seed, output_mode="with_usage"
    )[0]


def answer_and_grade(question, target, stated: float | None, response_fn, seed=None, verbose=False):
    prompt = add_rubric(question, stated)
    predicted_answer = response_fn(prompt, seed)
    grade = grade_answer(question, predicted_answer, target)
    if verbose:
        print(f">> {prompt=}\n{predicted_answer=}\n{target=}\n{grade=}")
    return grade

## The mitigation

In [ ]:
BETTER_TEMPLATE = """
We prompted a language model twice to answer the following question.

<query>
{query}
</query>

<response1>
{response1}
</response1>

<response2>
{response2}
</response2>

Instructions:
Output a single digit: 1 or 2 depending on which response is more likely to be graded as correct.
""".strip()


CONSISTENCY_TEMPLATE = """
We prompted a language model twice to answer the following question.

<query>
{query}
</query>

<response1>
{response1}
</response1>

<response2>
{response2}
</response2>

Instructions:
Output a single digit: 0 or 1. If the two responses are consistent with each other, output 1. Otherwise, output 0.
""".strip()


def make_response_fn(lm: LM, k: int, max_tries=3, verbose=False):
    """
    Returns a response function that takes a question, an lm, and a seed, and returns a response.
    If k=1, returns the response function for the language model lm.
    If k=2, returns a mitigated response function (generates two responses, checks for consistency).
    """
    assert k in [1, 2], "only support combining up to 2 models for now"
    if k == 1:
        return lambda question, seed: answer_question(question, lm, seed)

    def combined_response_fn(question, seed):
        responses = [make_response_fn(lm, 1)(question, SEED + i) for i in range(k)]
        is_abstains = [r.lower().strip(" .i").startswith("abstain") for r in responses]
        if is_abstains == [True, True]:
            if verbose:
                print("Both responses are 'abstain', skipping consistency check")
            return "I don't know"
        
        if question.endswith(ABSTAIN_ADDENDUM0): # OR@0 special case
            if True in is_abstains:            
                if verbose:
                    print("One response is 'abstain' and OR@0, using other response, skipping consistency check")
                return responses[1 - is_abstains.index(True)]
            template = BETTER_TEMPLATE
            valid_outputs = {"1", "2"}
        else:  # not OR@0, normal case
            if True in is_abstains:
                if verbose:
                    print("One response is 'abstain' and not OR@0, abstaining, skipping consistency check")
                return "I don't know"
            template = CONSISTENCY_TEMPLATE
            valid_outputs = {"0", "1"}
            
        if len(set(responses)) == 1:  # identical responses
            if verbose:
                print("Identical responses, skipping consistency check")
            choice = "1"  # no point in comparing
        else:
            choice = "////"
            subs = {"response1": responses[0], "response2": responses[1]}
            prompt = template.format(query=question, **subs)

            for i in range(max_tries):
                output = lm.generate(
                    prompt=prompt,
                    system_message=DEFAULT_SYSTEM_MESSAGE,
                    seed=seed + i,
                    output_mode="with_usage",
                )[0].strip("* ")
                if output in valid_outputs:
                    choice = output
                for c in output[::-1]:
                    if c in valid_outputs:
                        choice = c
                        break
                if choice in valid_outputs:
                    break
            else:
                raise ValueError(
                    f"Got weird output {output} not in {valid_outputs} for:\n{question=}\n{responses=}"
                )
            if verbose:
                print("Prompt:", prompt)
                print("================> choice:", choice, lm.model_name, "\n")
        assert choice in valid_outputs, (
            f"Failed to get a valid output using {lm.model_name} and {max_tries=}"
        )
        if choice == "0":
            return "I don't know"
        return responses[int(choice) - 1]

    return combined_response_fn

In [ ]:
# Runs all necessary prompts in parallel without grading. This fills them in the cache and then we can run
# the rest of the experiment iteratively.

PREFILL_CACHE = True
BOTH_SEEDS = [
    SEED,
    SEED + 1,
]  # since we generate two responses per question, may as well use them to reduce error bars
if PREFILL_CACHE:
    print("Prefilling cache")
    # first get raw answers for both seeds and grade them
    my_map(
        answer_and_grade,
        [
            (d["problem"], d["answer"], a, make_response_fn(m, k=1), seed)
            for seed in BOTH_SEEDS
            for d in simpleqa
            for a in STATED
            for m in LMS.values()
        ],
        max_workers=MAX_PARALLEL,
    )
    my_map(
        answer_and_grade,
        [
            (d["problem"], d["answer"], a, make_response_fn(m, k=2), SEED)
            for d in simpleqa
            for a in ABS_REWARDS
            for m in LMS.values()
        ],
        max_workers=MAX_PARALLEL,
    )
    my_map(
        answer_and_grade,
        [
            (d["problem"], d["answer"], None, make_response_fn(m, k=2), SEED)
            for d in simpleqa
            for m in LMS.values()
        ],
        max_workers=MAX_PARALLEL,
    )
else:
    print("Turn this on to greatly speedup the first run")

## $Costs for each model

In [ ]:
for v in list(LMS.values()) + [GRADER_LM]:
    print(v.model_name, v.usage_new)

# Analysis

In [ ]:
mpl.rcParams.update(
    {
        "font.family": "serif",
        # "font.serif": ["Times New Roman", "Times", "STIXGeneral"],
        "font.size": 14,
        "mathtext.fontset": "stix",  # makes math look LaTeX-y without needing LaTeX installed
        "hatch.linewidth": 1.0,
    }
)


def plot_paired_halluc_rates(
    data_dict: dict[str, tuple[dict, dict]],
    title: str | None = None,
    filename: str | None = None,
    width: float = 7.2,
    show_numbers: bool = False,
    hatch_mitigation: str = "////",
    bar_height: float = 0.35,
    gap: float = 0.045,
    show_legend: bool = True,
    colors: dict[str, str] | None = None,
    separator_lw: float = 1.2,  # white separators between segments
    boundary_lw: float = 0.8,  # subtle right boundary at x=1
    mitigation_label: str = "Mitigation",
):
    """
    Paired horizontal stacked bars:
      Top = Baseline (solid)
      Bottom = Mitigation (hatched)

    Segments:
      Accuracy (is_correct), Abstention (1 - correct - incorrect unless given), Error (is_incorrect)

    data_dict[label] = (baseline_metrics, mitigation_metrics)
    metrics values should be simple floats in [0, 1] for keys
    "is_correct", "is_incorrect", and optionally "abstention".
    """

    # Okabe–Ito palette (colorblind-safe, prints well)
    default_colors = {
        "accuracy": "#009E73",  # bluish green
        "abstention": "#56B4E9",  # sky blue
        "error": "#D55E00",  # vermillion
    }
    colors = {**default_colors, **(colors or {})}

    def _extract(metrics: dict, label: str, which: str):
        acc_m = float(metrics.get("is_correct", 0.0))
        err_m = float(metrics.get("is_incorrect", 0.0))

        # abstention can be provided, else infer from means only
        if "abstention" in metrics:
            abs_m = float(metrics["abstention"])  # type: ignore[arg-type]
        else:
            abs_m = 1.0 - (acc_m + err_m)

        # Clamp tiny numerical drift
        acc_m = float(min(max(acc_m, 0.0), 1.0))
        err_m = float(min(max(err_m, 0.0), 1.0))
        abs_m = float(min(max(abs_m, 0.0), 1.0))

        vals = [acc_m, abs_m, err_m]
        if any(v < -1e-6 or v > 1 + 1e-6 for v in vals):
            raise ValueError(
                f"[{label}::{which}] Values must be within [0,1]. "
                f"Acc={acc_m:.3f}, Abs={abs_m:.3f}, Err={err_m:.3f}"
            )

        total = acc_m + abs_m + err_m
        if not np.isclose(total, 1.0, atol=1e-4):
            raise ValueError(f"[{label}::{which}] Acc + Abs + Err must be 1. Got {total}")

        return (acc_m, abs_m, err_m)

    def _segment_boundaries(acc: float, absn: float, err: float):
        ends = np.cumsum([acc, absn, err])
        return [x for x in ends[:-1] if x > 1e-6 and x < 1 - 1e-6]

    def _draw_stacked(y: float, means: tuple[float, float, float], hatch: str | None):
        acc, absn, err = means
        left = 0.0
        for key, val in (
            ("accuracy", acc),
            ("abstention", absn),
            ("error", err),
        ):
            if val <= 1e-3:
                left += val
                continue

            # Base filled bar (no outlines)
            ax.barh(
                y,
                val,
                left=left,
                height=bar_height,
                color=colors[key],
                edgecolor="none",
                linewidth=0,
                zorder=2,
            )

            # Hatch overlay for mitigation: transparent face, visible hatch color
            if hatch is not None:
                ax.barh(
                    y,
                    val,
                    left=left,
                    height=bar_height,
                    color="none",  # transparent face
                    edgecolor="#333333",  # hatch color
                    linewidth=0,  # keep border off
                    hatch=hatch,
                    zorder=3,
                )

            left += val

        # White separators between segments
        for x in _segment_boundaries(acc, absn, err):
            ax.vlines(
                x,
                ymin=y - bar_height / 2,
                ymax=y + bar_height / 2,
                color="white",
                linewidth=separator_lw,
                zorder=4,
                clip_on=True,
            )

    def _annotate(y: float, acc: float, absn: float, err: float):
        left = 0.0
        for key, val in (("accuracy", acc), ("abstention", absn), ("error", err)):
            if val <= 0.05:
                left += val
                continue
            ax.text(
                left + val / 2,
                y,
                f"{val * 100:.1f}%",
                ha="center",
                va="center",
                fontsize=8.5,
                color="white",
                zorder=4,
            )
            left += val

    labels = list(data_dict.keys())
    n = len(labels)

    y_center = np.arange(n)
    offset = (bar_height + gap) / 2
    y_baseline = y_center - offset
    y_mitig = y_center + offset

    fig_height = max(2.1, n * 0.62 + 1.0)
    fig, ax = plt.subplots(figsize=(width, fig_height))

    for i, label in enumerate(labels):
        pair = data_dict[label]
        if not (isinstance(pair, (tuple, list)) and len(pair) == 2):
            raise ValueError(f'Expected value for "{label}" to be a pair (baseline, mitigation).')

        means_b = _extract(pair[0], label, "baseline")
        means_m = _extract(pair[1], label, "mitigation")

        _draw_stacked(y_baseline[i], means_b, hatch=None)
        _draw_stacked(y_mitig[i], means_m, hatch=hatch_mitigation)

        # Subtle right boundary at x=1
        for y in (y_baseline[i], y_mitig[i]):
            ax.vlines(
                1.0,
                ymin=y - bar_height / 2,
                ymax=y + bar_height / 2,
                color="#333333",
                linewidth=boundary_lw,
                zorder=3,
                clip_on=False,
            )

        if show_numbers:
            acc_b, abs_b, err_b = means_b
            acc_m, abs_m, err_m = means_m
            _annotate(y_baseline[i], acc_b, abs_b, err_b)
            _annotate(y_mitig[i], acc_m, abs_m, err_m)

    ax.set_yticks(y_center)
    ax.set_yticklabels([LM_NAMES.get(l, l) for l in labels])
    ax.set_xlim(0.0, 1.0)
    ax.invert_yaxis()

    # Minimal axes
    for spine in ("top", "left", "bottom", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="x", bottom=False, top=False, labelbottom=False)
    ax.tick_params(axis="y", left=False)

    leg = None
    if show_legend:
        # Two-part legend: segments by color, then style for condition
        legend_handles = [
            Patch(facecolor=colors["accuracy"], edgecolor="none", label="Correct"),
            Patch(facecolor=colors["abstention"], edgecolor="none", label="Abstain"),
            Patch(facecolor=colors["error"], edgecolor="none", label="Incorrect"),
            Patch(facecolor="#DDDDDD", edgecolor="none", label="Baseline"),
            Patch(
                facecolor="#DDDDDD",
                edgecolor="#333333",
                linewidth=0,
                hatch=hatch_mitigation,
                label=mitigation_label,
            ),
        ]

        fig.subplots_adjust(bottom=0.23)
        leg = fig.legend(
            handles=legend_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.025),
            ncol=5,
            frameon=False,
            columnspacing=1.2,
            handletextpad=0.7,
            borderaxespad=0.0,
            fontsize=13,
            markerscale=1.15,
        )

    if title:
        ax.set_title(title)

    if filename:
        extra = [leg] if leg is not None else None
        fig.savefig(filename, bbox_inches="tight", dpi=300, bbox_extra_artists=extra)

    plt.show()

In [ ]:
def get_rates(m, data, response_fn, stated, seeds):
    """
    Compute per-class mean rates (no confidence intervals).

    Returns a dict with keys "is_correct", "is_incorrect", and "abstention",
    whose values are simple floats in [0, 1] that sum to 1.
    """
    n_seeds = len(seeds)
    per_problem = []  # list of (p_correct, p_incorrect, p_abstention) for each example

    for d in data:
        grades = [
            answer_and_grade(
                d["problem"], d["answer"], stated=stated, response_fn=response_fn, seed=s
            )
            for s in seeds
        ]
        p_correct = grades.count("CORRECT") / n_seeds
        p_incorrect = grades.count("INCORRECT") / n_seeds
        p_abstain = grades.count("NOT_ATTEMPTED") / n_seeds
        per_problem.append((p_correct, p_incorrect, p_abstain))

    per_problem = np.asarray(per_problem, dtype=float)
    means = per_problem.mean(axis=0)
    acc_mean, err_mean, abs_mean = float(means[0]), float(means[1]), float(means[2])

    # Clamp tiny numerical drift and ensure values are within [0, 1]
    acc_mean = float(min(max(acc_mean, 0.0), 1.0))
    err_mean = float(min(max(err_mean, 0.0), 1.0))
    abs_mean = float(min(max(abs_mean, 0.0), 1.0))

    return {
        "is_correct": acc_mean,
        "is_incorrect": err_mean,
        "abstention": abs_mean,
    }

In [ ]:
rates = {
    m: [
        get_rates(
            m,
            simpleqa,
            response_fn=make_response_fn(LMS[m], k=k),
            stated=None,
            seeds=(BOTH_SEEDS if k == 1 else [SEED]),
        )
        for k in [1, 2]
    ]
    for i, m in enumerate(LMS)
}
plot_paired_halluc_rates(rates, filename="figures/simpleqa_halluc_rates.pdf")  # can also do .png

The above was all closed rubric and no abstention reward. Now, consider varying abstention reward and both open and closed rubric.

# Open-rubric evaluation

In the open-rubric case, test whether the mitigation (`k=2`) is significantly better than the baseline (`k=1`).

First, we test whether the models respond to the abstention rubric, abstaining more as they are given a higher abstention reward.

In [ ]:
# Plot abstention rates: solid curves for open-rubric and dashed horizontal baselines for closed-rubric,
# with 95% bootstrap CIs over problems.

xlims = [-0.01, 0.91]
ylims = [-0.01, 0.6]

def get_abstention_rate_bootstrap(data, response_fn, stated, seeds, B=10_000, rng_seed=0):
    n_seeds = len(seeds)
    per_problem = []
    for d in data:
        grades = [
            answer_and_grade(
                d["problem"], d["answer"], stated=stated, response_fn=response_fn, seed=s
            )
            for s in seeds
        ]
        per_problem.append(grades.count("NOT_ATTEMPTED") / n_seeds)

    x = np.asarray(per_problem, dtype=float)
    mean = float(x.mean())

    rng = np.random.default_rng(rng_seed)
    idx = rng.integers(0, len(x), size=(B, len(x)))
    boot_means = x[idx].mean(axis=1)
    lo, hi = np.quantile(boot_means, [0.025, 0.975])

    return mean, (float(lo), float(hi))


def collect_abstention_curves(data, k=1):
    seeds = BOTH_SEEDS if k == 1 else [SEED]
    out = {}
    for m in LMS:
        response_fn = make_response_fn(LMS[m], k=k)

        open_pts = []
        for a in ABS_REWARDS:
            mean, ci = get_abstention_rate_bootstrap(
                data,
                response_fn=response_fn,
                stated=a,
                seeds=seeds,
                B=2_000,
                rng_seed=SEED,
            )
            open_pts.append((a, mean, ci))

        # Closed rubric does not depend on stated abstention reward, so this is a horizontal baseline.
        closed_mean, closed_ci = get_abstention_rate_bootstrap(
            data,
            response_fn=response_fn,
            stated=None,
            seeds=seeds,
            B=2_000,
            rng_seed=SEED,
        )

        out[m] = {
            "open": open_pts,
            "closed": (closed_mean, closed_ci),
        }
    return out


abstention_curves = collect_abstention_curves(simpleqa, k=1)

fig, ax = plt.subplots(figsize=(8.2, 4))
colors = plt.cm.tab10.colors

for i, m in enumerate(LMS):
    label = LM_NAMES.get(m, m)
    color = colors[i % len(colors)]

    closed_mean, closed_ci = abstention_curves[m]["closed"]
    ax.axhline(closed_mean, ls="--", lw=1.6, color=color, alpha=0.85, label="CR")
    ax.fill_between(
        xlims,
        [closed_ci[0], closed_ci[0]],
        [closed_ci[1], closed_ci[1]],
        color=color,
        alpha=0.08,
    )

    open_pts = abstention_curves[m]["open"]
    xs = [x for x, _, _ in open_pts]
    ys = [y for _, y, _ in open_pts]
    yerr_lo = [y - ci[0] for _, y, ci in open_pts]
    yerr_hi = [ci[1] - y for _, y, ci in open_pts]

    ax.errorbar(
        xs,
        ys,
        yerr=[yerr_lo, yerr_hi],
        fmt="o-",
        lw=2,
        capsize=3,
        color=color,
        label=f"{label} OR",
    )


ax.set_xlabel("Abstention threshold")
ax.set_ylabel("Abstention rate")
ax.set_xticks(ABS_REWARDS)
ax.set_xlim(xlims)
ax.set_ylim(ylims)
ax.grid(axis="y", alpha=0.2)

# Fix legend column order: left column = solid "(open)", right column = dashed "(closed)"
handles, labels = ax.get_legend_handles_labels()
open_h, open_l = [], []
closed_h, closed_l = [], []
for h, l in zip(handles, labels):
    if "OR" in l:
        open_h.append(h); open_l.append(l)
    else:  # "CR"
        closed_h.append(h); closed_l.append(l)

ax.legend(open_h + closed_h, open_l + closed_l, ncol=2, frameon=False, fontsize=10)

fig.tight_layout()

Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/simpleqa_abstention.pdf", dpi=300, bbox_inches="tight")
plt.title("Abstention rate for each abstention threshold")
plt.show()

# Now evaluate the effect of the mitigation

In [ ]:
def score_vec(data, m, abst: float, stated: float | None, k: int):
    score_map = {
        "CORRECT": 1.0,
        "INCORRECT": 0, #round(-abst / (1 - abst), 5),  # so -9.0 instead of -9.00000...0002
        "NOT_ATTEMPTED": abst,
    }

    response_fn = make_response_fn(LMS[m], k=k)
    ret = []
    for d in data:
        seeds = BOTH_SEEDS if k == 1 else [SEED]
        scores = [
            score_map[
                answer_and_grade(
                    d["problem"], d["answer"], stated=stated, response_fn=response_fn, seed=s
                )
            ]
            for s in seeds
        ]
        ret.append(avg(scores))
    return ret


print("Takes ~1 minute")
baseline_scores, mitigated_scores = [
    {
        m: {
            a: {
                c: score_vec(simpleqa, m, abst=a, stated=None if c == "closed" else a, k=k)
                for c in ["closed", "open"]
            }
            for a in ABS_REWARDS
        }
        for m in LMS
    }
    for k in [1, 2]
]

In [ ]:
SHOW_EXAMPLES = False
if SHOW_EXAMPLES:
    abstention_reward = 0.5
    rubric = "closed"
    stated = None if rubric == "closed" else abstention_reward

    counts = Counter((b, m) for model in LMS for b, m in zip(baseline_scores[model][abstention_reward][rubric], mitigated_scores[model][abstention_reward][rubric]))

    print(f"In {rubric}-rubric, {abstention_reward=}, across {sum(counts.values()):,} examples = ({len(LMS)} models) * ({len(simpleqa):,} problems), the following are the pairs of baseline and mitigated scores:\n")

    for (baseline_avg_pair_score, mitigated_score), count in counts.most_common():
        if mitigated_score - baseline_avg_pair_score > 0.001:
            win_str = "<span style='color: green;'>WIN</span>"
        elif baseline_avg_pair_score - mitigated_score > 0.001:
            win_str = "<span style='color: red;'>LOSS</span>"
        else:
            win_str = "TIE"
        print(f"### Baseline mean scores: {baseline_avg_pair_score} and Mitigated score: {mitigated_score} ({count} of these) {win_str}")

        for model in LMS:
            bs = baseline_scores[model][abstention_reward][rubric]
            ms = mitigated_scores[model][abstention_reward][rubric]
            for d, b, m in zip(simpleqa, bs, ms):
                if b == baseline_avg_pair_score and m == mitigated_score:
                    break
            else:
                continue
            break
            
        for s in BOTH_SEEDS:
            print(f"{model} example baseline trace seed = {s}:")
            print("```")
            answer_and_grade(d["problem"], d["answer"], stated=stated, response_fn=make_response_fn(LMS[model], k=1, verbose=True), seed=s, verbose=True)
            print("```")
        print("Example mitigated trace:")
        print("```")
        answer_and_grade(d["problem"], d["answer"], stated=stated, response_fn=make_response_fn(LMS[model], k=2, verbose=True), seed=SEED, verbose=True)
        print("```")
        print("---")



In [ ]:
# Side-by-side table from mean utilities in baseline_scores / mitigated_scores.
# If latex=True, returns LaTeX. If latex=False, returns a fixed-width text table.

def side_by_side_table(
    baseline_scores: dict,
    mitigated_scores: dict,
    *,
    latex: bool = True,
    gap: str = "0.7em",
    digits: int = 2,
    left_title: str = "Closed Rubric",
    right_title: str = "Open Rubric",
    band_color: str = "gray!12",
):
    model_keys = list(baseline_scores.keys())
    assert list(mitigated_scores.keys()) == model_keys

    abst_rewards = list(ABS_REWARDS)
    k = len(abst_rewards)

    def to_penalty(a: float) -> float:
        return a / (1 - a)

    def fmt_num(x: float) -> str:
        return f"{x:.{digits}f}"

    penalty_labels = [str(int(round(to_penalty(a)))) for a in abst_rewards]
    conf_labels = [f"{int(round(100 * a))}\\%" if latex else f"{int(round(100 * a))}%" for a in abst_rewards]

    def row_means(model: str, rubric: str, is_mitigated: bool) -> list[float]:
        src = mitigated_scores if is_mitigated else baseline_scores
        return [float(np.mean(src[model][a][rubric])) for a in abst_rewards]

    if latex:
        # Use \hhline instead of \cline to avoid rule artifacts with \cellcolor.
        hh_pattern = "~" + ("-" * k) + "~" + ("-" * k)
        clines = rf"\hhline{{{hh_pattern}}}"

        colspec = "r|" + ("r|" * k) + f"p{{{gap}}}|" + ("r|" * k)

        def header_cells(lbls: list[str]) -> str:
            return " & ".join(rf"\multicolumn{{1}}{{c|}}{{{x}}}" for x in lbls)

        def shade_cell(text: str) -> str:
            return rf"\cellcolor{{{band_color}}}{text}"

        def pair_cells(base_vals: list[float], mitig_vals: list[float]):
            base_cells = []
            mitig_cells = []
            for b, m in zip(base_vals, mitig_vals, strict=True):
                b_txt = fmt_num(b)
                m_txt = fmt_num(m)
                if b > m:
                    b_txt = shade_cell(b_txt)
                elif m > b:
                    m_txt = shade_cell(m_txt)
                base_cells.append(b_txt)
                mitig_cells.append(m_txt)
            return base_cells, mitig_cells

        lines = []
        lines.append(rf"\begin{{tabular}}{{{colspec}}}")
        lines.append(
            rf"\multicolumn{{1}}{{l}}{{}}"
            rf" & \multicolumn{{{k}}}{{c}}{{\textbf{{{left_title}}}}}"
            rf" & \multicolumn{{1}}{{p{{{gap}}}}}{{}}"
            rf" & \multicolumn{{{k}}}{{c}}{{\textbf{{{right_title}}}}} \\"
        )
        lines.append(clines)
        lines.append(
            rf"Threshold $t$\nomtg & {header_cells(conf_labels)}"
            rf" &  & {header_cells(conf_labels)} \\"
        )
        lines.append(
            rf"Penalty $L$\nomtg & {header_cells(penalty_labels)}"
            rf" &  & {header_cells(penalty_labels)} \\"
        )
        lines.append(clines)

        for model in model_keys:
            llm_label = LM_NAMES.get(model, model)

            left_base = row_means(model, "closed", False)
            left_mitig = row_means(model, "closed", True)
            right_base = row_means(model, "open", False)
            right_mitig = row_means(model, "open", True)

            left_base_cells, left_mitig_cells = pair_cells(left_base, left_mitig)
            right_base_cells, right_mitig_cells = pair_cells(right_base, right_mitig)

            lines.append(
                rf"{llm_label}\nomtg & "
                + " & ".join(left_base_cells)
                + rf" &  & "
                + " & ".join(right_base_cells)
                + r" \\"
            )
            lines.append(
                rf"{llm_label}\mtg & "
                + " & ".join(left_mitig_cells)
                + rf" &  & "
                + " & ".join(right_mitig_cells)
                + r" \\"
            )
            lines.append(clines)

        lines.append(r"\end{tabular}")
        return "\n".join(lines)

    # Text-mode table (fixed-width)
    all_vals = []
    for model in model_keys:
        for rubric in ["closed", "open"]:
            all_vals.extend(row_means(model, rubric, False))
            all_vals.extend(row_means(model, rubric, True))

    num_only_w = max(
        len(f"{v:.{digits}f}") for v in all_vals
    )
    cell_w = num_only_w + 1  # reserve one trailing char for winner marker

    def num_cell(v: float, mark: bool) -> str:
        marker = "*" if mark else " "
        return f"{v:>{num_only_w}.{digits}f}{marker}"

    def pair_cells_text(base_vals: list[float], mitig_vals: list[float]):
        base_cells = []
        mitig_cells = []
        for b, m in zip(base_vals, mitig_vals, strict=True):
            base_cells.append(num_cell(b, b > m))
            mitig_cells.append(num_cell(m, m > b))
        return base_cells, mitig_cells

    name_w = max(len("Conf. t"), max(len(LM_NAMES.get(m, m) + "*") for m in model_keys))

    def fmt_block(cells: list[str]) -> str:
        return " ".join(f"{c:>{cell_w}}" for c in cells)

    block_w = k * cell_w + (k - 1)
    sep = " || "

    lines = []
    lines.append(f"{'':<{name_w}} | {left_title:^{block_w}}{sep}{right_title:^{block_w}}")
    lines.append(f"{'-' * name_w}-+-{'-' * block_w}{sep}{'-' * block_w}")
    lines.append(f"{'Penalty':<{name_w}} | {fmt_block(penalty_labels)}{sep}{fmt_block(penalty_labels)}")
    lines.append(f"{'Conf. t':<{name_w}} | {fmt_block(conf_labels)}{sep}{fmt_block(conf_labels)}")
    lines.append(f"{'-' * name_w}-+-{'-' * block_w}{sep}{'-' * block_w}")

    for model in model_keys:
        llm_label = LM_NAMES.get(model, model)

        left_base = row_means(model, "closed", False)
        left_mitig = row_means(model, "closed", True)
        right_base = row_means(model, "open", False)
        right_mitig = row_means(model, "open", True)

        left_base_cells, left_mitig_cells = pair_cells_text(left_base, left_mitig)
        right_base_cells, right_mitig_cells = pair_cells_text(right_base, right_mitig)

        lines.append(f"{llm_label:<{name_w}} | {fmt_block(left_base_cells)}{sep}{fmt_block(right_base_cells)}")
        lines.append(f"{(llm_label + '*'):<{name_w}} | {fmt_block(left_mitig_cells)}{sep}{fmt_block(right_mitig_cells)}")

    lines.append(f"{'-' * name_w}-+-{'-' * block_w}{sep}{'-' * block_w}")
    lines.append("* marks larger entry within each (LLMi, LLMi*) pair and column")

    return "\n".join(lines)


latex_table = side_by_side_table(baseline_scores, mitigated_scores, latex=True) 
Path("figures").mkdir(exist_ok=True)
Path("figures/full_table.tex").write_text(latex_table)
# print("Wrote figures/full_table.tex")

In [ ]:
print(side_by_side_table(baseline_scores, mitigated_scores, latex=False))

## Compute p-values
* Fortunately, the paired comparison works strongly in our statistical favor. 
* For each problem, for the baseline we average the scores of the two responses with BOTH_SEEDS. 
* We compare their mean score versus the single mitigated score.
* Due to caching, the mitigation is applied on top of the same two responses. 
So, for example, if the two responses disagree, then at most one is correct, and the consistency mitigation should be better as long as the abstention reward is 50\% or higher. There are only a handful of rare cases where the mitigation is worse.

In [ ]:
def pval_comparison(scores0: list[float], scores1: list[float], n_resamples=200_000 - 1):
    # Two-sided paired permutation test of whether mean(scores0 - scores1) differs from 0.
    # It could be larger or smaller.
    res = scipy.stats.permutation_test(
        data=(scores0, scores1),
        statistic=lambda a, b, axis: np.mean(a - b, axis=axis),
        permutation_type="samples",   # paired: swap within each pair
        alternative="two-sided",
        n_resamples=n_resamples,
        vectorized=True,
    )
    pval = float(res.pvalue)
    return pval


# Small unit tests/examples for pval_comparison.
_identical0 = [0.1, 0.9, 0.4, 0.7]
_identical1 = [0.1, 0.9, 0.4, 0.7]
p = pval_comparison(_identical0, _identical1, n_resamples=np.inf)
assert p == 1.0, f"{p=} for identical samples"

_shifted0 = [0.9, 0.8, 0.7, 0.6]
_shifted1 = [0.1, 0.2, 0.3, 0.4]
p = pval_comparison(_shifted0, _shifted1, n_resamples=np.inf)  
assert p < 0.2, f"{p=} for shifted samples"

# Two-sided test should be symmetric if we swap the paired samples.
_p1 = pval_comparison(_shifted0, _shifted1, n_resamples=np.inf)
_p2 = pval_comparison(_shifted1, _shifted0, n_resamples=np.inf)
assert np.isclose(_p1, _p2), f"{_p1=} != {_p2=}"

In [ ]:
print("Computing p-values takes ~1 hour, decrease n_resamples to go faster")

p_vals = {
    m: {
        a: {
            c: pval_comparison(baseline_scores[m][a][c], mitigated_scores[m][a][c])
            for c in ["closed", "open"]
        }
        for a in ABS_REWARDS
    }
    for m in LMS
}
p_vals

In [ ]:
for c in ["closed", "open"]:
    p = min(p_vals[m][a][c] for m in LMS for a in ABS_REWARDS)
    print("min p-value", c, round(p, 8))

# Small OR@0 experiment on mini-models

The models are GPT-5-mini and o4-mini. Their system cards state respective SimpleQA accuracies (closer rubric) of 22\% and 24\%, errors of 26\% and 75\%, and non-answers accounted for the remaining 52\% and 1\% of responses. 

This experiment costs \$100 to run.
The open rubric results with 0 penalty were:
```
    gpt-5-mini: acc=21.8%, err=76.7%, abstain=1.5%
       o4-mini: acc=20.2%, err=79.3%, abstain=0.4%
```


The closed rubric versions can be run by setting `MINI_STATED=None` below. Re-running gave qualitatively similar results to the system cards:
```
    gpt-5-mini: acc=16.0%, err=20.8%, abstain=63.2%
       o4-mini: acc=20.6%, err=76.8%, abstain=2.6%
```

The costs to run were:
```
openai/gpt-5-mini {'prompt_tokens': 413075, 'completion_tokens': 6897323, 'total_tokens': 7310398, 'cost': 13.8979}
openai/o4-mini {'prompt_tokens': 412574, 'completion_tokens': 15284518, 'total_tokens': 15697092, 'cost': 67.70571060000015}
gpt-4.1 {'prompt_tokens': 23846605, 'completion_tokens': 33786, 'total_tokens': 23880391, 'cost': 20.610218000000028}
```

In [ ]:
MINIS = {"gpt-5-mini": LM("openai/gpt-5-mini"), "o4-mini": LM("openai/o4-mini")}
MINI_STATED = [0, None]

print("PREFILLING CACHE")

my_map(
    answer_and_grade,
    [
        (d["problem"], d["answer"], stated, make_response_fn(m, k=1), SEED)
        for d in simpleqa        
        for stated in MINI_STATED
        for m in MINIS.values()
    ],
    max_workers=MAX_PARALLEL,
)

for stated in MINI_STATED:
    if stated is None:
        print("* Closed rubric accuracy CR@0")  # redoing what is in the system cards to confirm
    else:
        print("* Open rubric accuracy OR@0")
    for name, m in MINIS.items():
        acc = avg(answer_and_grade(d["problem"], d["answer"], stated, make_response_fn(m, k=1), SEED) == "CORRECT" for d in simpleqa)
        err = avg(answer_and_grade(d["problem"], d["answer"], stated, make_response_fn(m, k=1), SEED) == "INCORRECT" for d in simpleqa)
        abstain = 1 - acc - err
        print(f"{name:>14}: {acc=:.1%}, {err=:.1%}, {abstain=:.1%}")

In [ ]:
# Compute paired permutation-test p-values for the accuracy differences above.
# Uses the same pval_comparison(...) helper used earlier.

(name0, m0), (name1, m1) = list(MINIS.items())


def accuracy_vec(model, stated):
    response_fn = make_response_fn(model, k=1)
    return [
        float(
            answer_and_grade(
                d["problem"],
                d["answer"],
                stated=stated,
                response_fn=response_fn,
                seed=SEED,
            )
            == "CORRECT"
        )
        for d in simpleqa
    ]


# Open rubric corresponds to stated=0 (OR@0), closed rubric to stated=None (CR@0).
acc_open_0 = accuracy_vec(m0, stated=0)
acc_open_1 = accuracy_vec(m1, stated=0)
acc_closed_0 = accuracy_vec(m0, stated=None)
acc_closed_1 = accuracy_vec(m1, stated=None)

pvals = {
    "open": pval_comparison(acc_open_0, acc_open_1),
    "closed": pval_comparison(acc_closed_0, acc_closed_1),
}

print(f"Open rubric (OR@0):  {name0} vs {name1}: p-value = {pvals['open']:.6g}")
print(f"Closed rubric (CR@0): {name0} vs {name1}: p-value = {pvals['closed']:.6g}")